<a href="https://colab.research.google.com/github/Bokenasubi-T/pomacea-distribution-jp/blob/main/04_WorldClim.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

04. Extract WorldClim climate data for *Pomacea canaliculata* grid cells

- Input: `data/pomacea_jp_thinned_30s.csv`, `data/pomacea_jp_thinned_5min.csv`, WorldClim v2.1 bioclimatic variables (30 s and 5 min)
- Output: `data/pomacea_jp_thinned_30s_climate.csv`, `data/pomacea_jp_thinned_5min_climate.csv`

- Variables: bio1 (annual mean temperature), bio6 (min temperature of coldest month), bio12 (annual precipitation)
- Note: WorldClim rasters are downloaded and cropped to Japan in this notebook; they are not stored in the repository.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

# Project folders on Google Drive
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/pomacea"
DATA_DIR = f"{BASE_DIR}/data"

Mounted at /content/drive


In [5]:
#install library for reading and writing raster data
!pip install rasterio

In [10]:
# Download WorldClim v2.1 bioclim(30 s) to Colab's temporary disk, not Drive(the zip is several GB)
!wget -nc -P /content/worldclim https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_30s_bio.zip

# Unzip only the variables we use: bio1, bio6, bio12
!unzip -n -j /content/worldclim/wc2.1_30s_bio.zip wc2.1_30s_bio_1.tif wc2.1_30s_bio_6.tif wc2.1_30s_bio_12.tif -d /content/worldclim

--2026-09-26 09:34:07--  https://geodata.ucdavis.edu/climate/worldclim/2_1/base/wc2.1_30s_bio.zip
Resolving geodata.ucdavis.edu (geodata.ucdavis.edu)... 128.120.146.30
Connecting to geodata.ucdavis.edu (geodata.ucdavis.edu)|128.120.146.30|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10401038857 (9.7G) [application/zip]
Saving to: ‘/content/worldclim/wc2.1_30s_bio.zip’

wc2.1_30s_bio.zip   100%[===================>]   9.69G  26.7MB/s    in 7m 29s  

2026-09-26 09:41:36 (22.1 MB/s) - ‘/content/worldclim/wc2.1_30s_bio.zip’ saved [10401038857/10401038857]

Archive:  /content/worldclim/wc2.1_30s_bio.zip
  inflating: /content/worldclim/wc2.1_30s_bio_1.tif  
  inflating: /content/worldclim/wc2.1_30s_bio_12.tif  
  inflating: /content/worldclim/wc2.1_30s_bio_6.tif  


In [15]:
import rasterio
from rasterio.windows import from_bounds

#Bounding box of Japan: (lon_min, lat_min, lon_max, lat_max)
BOUNDS = (122, 24, 146, 46)

for v in [1,6,12]:
  src_path = f"/content/worldclim/wc2.1_30s_bio_{v}.tif"
  out_path = f"{DATA_DIR}/wc2.1_30s_bio_{v}_japan.tif"

  with rasterio.open(src_path) as src:
    # Convert the bounding box to a pixel window and read only that part
    window = from_bounds(*BOUNDS, transform=src.transform).round_offsets().round_lengths()
    arr = src.read(1,window=window)
    profile = src.profile.copy()
    profile.update(
      height=arr.shape[0],
      width=arr.shape[1],
      transform=src.window_transform(window),
      compress="deflate"
    )

  with rasterio.open(out_path, "w", **profile) as dst:
   dst.write(arr,1)

  print(out_path, arr.shape)


/content/drive/MyDrive/Colab Notebooks/pomacea/data/wc2.1_30s_bio_1_japan.tif (2640, 2880)


/content/drive/MyDrive/Colab Notebooks/pomacea/data/wc2.1_30s_bio_6_japan.tif (2640, 2880)


/content/drive/MyDrive/Colab Notebooks/pomacea/data/wc2.1_30s_bio_12_japan.tif (2640, 2880)


In [16]:
import pandas as pd

pts = pd.read_csv(f"{DATA_DIR}/pomacea_jp_thinned_30s.csv")
coords = list(zip(pts["cell_lon"], pts["cell_lat"]))  # the order is (longitude, latitude)

with rasterio.open(f"{DATA_DIR}/wc2.1_30s_bio_6_japan.tif") as src:
    pts["bio6"] = [v[0] for v in src.sample(coords)]
    nodata = src.nodata

pts.loc[pts["bio6"] == nodata, "bio6"] = float("nan")
print(pts["bio6"].isna().sum(), "cells have NaN values")

24 cells have NaN values


In [17]:
# Show the cells with NaN values
nan_pts = pts[pts["bio6"].isna()]
print(nan_pts[["cell_lon", "cell_lat"]])

       cell_lon   cell_lat
35   136.529167  34.737500
38   139.354167  35.312500
50   130.462500  33.745833
84   130.195833  32.854167
85   130.429167  33.679167
87   129.779167  32.604167
95   130.495833  32.354167
97   131.862500  33.204167
100  131.987500  32.895833
105  130.904167  33.895833
106  130.779167  31.579167
110  130.812500  31.679167
112  130.695833  31.495833
113  130.795833  31.695833
115  130.237500  32.620833
119  131.620833  33.370833
122  131.737500  33.529167
126  129.945833  33.529167
127  131.470833  31.745833
136  130.604167  33.879167
145  134.854167  34.495833
147  131.087500  33.637500
148  130.629167  33.887500
150  129.645833  33.020833


In [18]:
import folium

# Base map with GSI pale tiles (OSM tiles are blocked in Colab)
m = folium.Map(location=[33, 131], zoom_start=6, tiles=None)
folium.TileLayer(
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html">地理院タイル</a>',
    name="GSI pale map",
).add_to(m)

HALF = 1 / 240  # half the width of a 30-arc-second cell in degrees

# Cells with NaN values (red point + cell boundary)
nan_layer = folium.FeatureGroup(name="Cells with NaN")
for _, r in nan_pts.iterrows():
    lat, lon = r["cell_lat"], r["cell_lon"]
    folium.Rectangle(
        bounds=[[lat - HALF, lon - HALF], [lat + HALF, lon + HALF]],
        color="red", weight=1, fill=True, fill_opacity=0.2,
    ).add_to(nan_layer)
    folium.CircleMarker(
        location=[lat, lon],
        radius=4, color="red", fill=True, fill_opacity=0.9,
        tooltip=f"lon {lon:.4f}, lat {lat:.4f}",  # show coordinates on hover
    ).add_to(nan_layer)
nan_layer.add_to(m)

# Save as HTML so the layer buttons work outside Colab too
m.save(f"{BASE_DIR}/outputs/pomacea_jp_nan_cells_map.html")
m

In [20]:
import numpy as np

def nearest_valid_cell(src, arr, lon, lat, radius=1):
    """Return (value, lon, lat) of the nearest non-NaN cell within `radius` cells."""
    row, col = src.index(lon, lat)          # convert lon/lat to row/col in the raster
    mask = np.ma.getmaskarray(arr)          # True where the cell has no data (sea)
    best, best_d = (np.nan, np.nan, np.nan), np.inf
    for dr in range(-radius, radius + 1):
        for dc in range(-radius, radius + 1):
            r, c = row + dr, col + dc
            inside = 0 <= r < arr.shape[0] and 0 <= c < arr.shape[1]
            if inside and not mask[r, c]:
                d = dr**2 + dc**2           # squared distance in cells
                if d < best_d:
                    x, y = src.xy(r, c)     # lon/lat of the cell center
                    best, best_d = (float(arr[r, c]), x, y), d
    return best

filled = pts[pts["bio6_filled"]]

with rasterio.open(f"{DATA_DIR}/wc2.1_30s_bio_6_japan.tif") as src:
    arr = src.read(1, masked=True)
    donors = [
        nearest_valid_cell(src, arr, lon, lat)
        for lon, lat in zip(filled["cell_lon"], filled["cell_lat"])
    ]

# Record which land cell each filled value came from (NaN for cells that were not filled)
pts["bio6_src_lon"] = np.nan
pts["bio6_src_lat"] = np.nan
pts.loc[pts["bio6_filled"], "bio6_src_lon"] = [d[1] for d in donors]
pts.loc[pts["bio6_filled"], "bio6_src_lat"] = [d[2] for d in donors]

print(pts.loc[pts["bio6_filled"], ["cell_lon", "cell_lat", "bio6_src_lon", "bio6_src_lat", "bio6"]])

       cell_lon   cell_lat  bio6_src_lon  bio6_src_lat  bio6
35   136.529167  34.737500    136.529167     34.745833   1.3
38   139.354167  35.312500    139.354167     35.320833   1.3
50   130.462500  33.745833    130.470833     33.745833   2.2
84   130.195833  32.854167    130.187500     32.854167   1.6
85   130.429167  33.679167    130.429167     33.687500   2.3
87   129.779167  32.604167    129.787500     32.604167   3.5
95   130.495833  32.354167    130.495833     32.362500   1.6
97   131.862500  33.204167    131.862500     33.212500   2.9
100  131.987500  32.895833    131.979167     32.895833   2.7
105  130.904167  33.895833    130.904167     33.887500   3.4
106  130.779167  31.579167    130.787500     31.579167   1.4
110  130.812500  31.679167    130.812500     31.687500   0.7
112  130.695833  31.495833    130.704167     31.495833   1.9
113  130.795833  31.695833    130.795833     31.704167   1.1
115  130.237500  32.620833    130.237500     32.629167   2.0
119  131.620833  33.3708

In [23]:
import folium

filled = pts[pts["bio6_filled"]]
HALF = 1 / 240  # half the width of a 30-arc-second cell in degrees

def cell_bounds(lon, lat):
    """Return the corners of a 30-arc-second cell for folium.Rectangle."""
    return [[lat - HALF, lon - HALF], [lat + HALF, lon + HALF]]

# Base map with GSI pale tiles
m = folium.Map(location=[filled["cell_lat"].mean(), filled["cell_lon"].mean()], zoom_start=6, tiles=None)
folium.TileLayer(
    tiles="https://cyberjapandata.gsi.go.jp/xyz/pale/{z}/{x}/{y}.png",
    attr='<a href="https://maps.gsi.go.jp/development/ichiran.html">地理院タイル</a>',
    name="GSI pale map",
).add_to(m)

for _, r in filled.iterrows():
    tip = f"bio6 = {r['bio6']:.1f} °C"

    # Original cell on the sea (red)
    folium.Rectangle(
        bounds=cell_bounds(r["cell_lon"], r["cell_lat"]),
        color="red", weight=1, fill=True, fill_opacity=0.25,
        tooltip=f"Original cell (no data)<br>{tip}",
    ).add_to(m)

    # Land cell whose value was used (blue)
    folium.Rectangle(
        bounds=cell_bounds(r["bio6_src_lon"], r["bio6_src_lat"]),
        color="blue", weight=1, fill=True, fill_opacity=0.25,
        tooltip=f"Source cell (land)<br>{tip}",
    ).add_to(m)

    # Line from the original cell to the source cell
    folium.PolyLine(
        locations=[[r["cell_lat"], r["cell_lon"]], [r["bio6_src_lat"], r["bio6_src_lon"]]],
        color="black", weight=1,
    ).add_to(m)

    # Center points drawn in screen pixels, so they stay visible at any zoom level
    folium.CircleMarker(
        location=[r["cell_lat"], r["cell_lon"]],
        radius=4, color="red", fill=True, fill_opacity=0.9,
        tooltip=f"Original cell (no data)<br>{tip}",
    ).add_to(m)
    folium.CircleMarker(
        location=[r["bio6_src_lat"], r["bio6_src_lon"]],
        radius=4, color="blue", fill=True, fill_opacity=0.9,
        tooltip=f"Source cell (land)<br>{tip}",
    ).add_to(m)

m.save(f"{BASE_DIR}/outputs/pomacea_jp_filled_cells_map.html")
m